# **Arabic Sentiment ML Project**

## **Import Libraries**

In [1]:
# Data handling & visualization
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# NLP preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer

# ML models & metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Sparse matrix operations
from scipy.sparse import hstack

# Misc
import string

## **Global Variables**

In [2]:
# Labels
LABELS = {"POS", "NEG", "OBJ", "NEUTRAL"}

# Arabic stopwords
nltk.download('stopwords')
arabic_stopwords = set(stopwords.words('arabic'))
negations = {"مش", "مو", "ما", "ليس", "لا", "لم", "لن", "بدون", "غير"}
arabic_stopwords -= negations

# Stemmer
stemmer = ISRIStemmer()

# Emoji dictionary
emoji_dict = {
    "😂": " EMO_POS ",
    "❤️": " EMO_POS ",
    "😡": " EMO_NEG ",
    "😢": " EMO_NEG "
}

# Arabic punctuation
ARABIC_PUNCTUATION = "؟،؛ـ"


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\batol\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## **1. Preprocessing**
We start by converting the raw TXT dataset into CSV for easier handling. 
We then normalize Arabic text to reduce variations in letters (e.g., "أ" → "ا") and remove elongations (like "مررررحبا" → "مرحبا"). 
Emojis are replaced with tokens representing sentiment, and negation words are prefixed with "NOT_" to capture their effect in sentiment analysis.

### **1.1 Dataset Loading & Conversion**

In [3]:
# Load TXT dataset and convert to CSV
def convert_to_csv(file_name="Arabic-Tweets_Dataset.txt"):
    data = []
    with open(file_name, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            parts = line.rsplit("\t", 1)
            if len(parts) == 2 and parts[1] in LABELS:
                text, label = parts
                data.append([text, label])
            else:
                print("Skipped malformed line:", line)
    df = pd.DataFrame(data, columns=["text", "label"])
    df["label"] = df["label"].replace({"NEUTRAL": "OBJ"})
    df.to_csv("Arabic-Tweets_Dataset.csv", index=False, encoding="utf-8-sig")
    return df

df = convert_to_csv()
df.head()

,text,label
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,OBJ


### **1.2 Normalize Arabic Letters**

In [4]:
# Arabic text normalization
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    return text

### **1.3 Remove Elongation**

In [5]:
def remove_elongation(text):
    return re.sub(r'(.)\1+', r'\1', text)

### **1.4 Replace Emojis**

In [6]:
def replace_emojis(text):
    for emoji, token in emoji_dict.items():
        text = text.replace(emoji, token)
    return text

### **1.5 Handle Negation**

In [7]:
def handle_negation(text):
    tokens = text.split()
    result = []
    i = 0
    while i < len(tokens):
        if tokens[i] in negations and i+1 < len(tokens):
            result.append("NOT_" + tokens[i+1])
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return " ".join(result)

### **1.6 Remove Stopwords**

In [8]:
def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in arabic_stopwords]
    return " ".join(words)

### **1.7 Apply Preprocessing**

In [9]:
def stem_text(text):
    words = text.split()
    return " ".join([stemmer.stem(w) for w in words])

def preprocess_text(text):
    text = normalize_arabic(text)
    text = remove_elongation(text)
    text = replace_emojis(text)
    text = handle_negation(text)
    text = remove_stopwords(text)
    text = stem_text(text)
    return text

df["cleaned_text"] = df["text"].apply(preprocess_text)
df.head()

,text,label,cleaned_text
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ,قال ريس #المحكمه_الدستوريه تظر قال #ريس_القضاء...
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS,اهن دكتور حمد جمل دين، قيد حزب صر، نسب صدر اول...
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG,ردع سقو بمر مرهاخر رسل عصم عري الي شنط شي قرف
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ,#الحريه_والعداله | شهد ان: #ليله_الاتحاديه اول...
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,OBJ,ولد اقل خطر حشش تضح اقل مل اله وكل تعط حضر فسق...


## **2. Feature Extraction**
We extract both:
1. **Text-based features** using TF(Term Frequency inside one tweet)-IDF(Inverse Document Frequency meaning in the whole document) (max 5000 features, unigrams and bigrams).
2. **Handcrafted features** specific to Arabic:
   - Negation count
   - Dialect words presence
   - Emoji features

These are combined using `hstack` into a single feature matrix `X`.


### **2.1 TF-IDF Representation**

In [10]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=3
    )
X_tfidf = tfidf.fit_transform(df["cleaned_text"])

### **2.2 Arabic-Specific Feature Functions**

In [11]:
def negation_count(text): 
    return sum(text.count(n) for n in negations)
def dialect_feature(text): 
    return int(any(w in text for w in ["مش","مو","شو","ليش","هيك"]))
def emoji_features(text):
    return pd.Series({"has_positive_emoji": int("EMO_POS" in text),
                      "has_negative_emoji": int("EMO_NEG" in text)})
df["neg_count"] = df["cleaned_text"].apply(negation_count)
df["dialect"] = df["cleaned_text"].apply(dialect_feature)
emoji_df = df["cleaned_text"].apply(emoji_features)
df = pd.concat([df, emoji_df], axis=1)

handcrafted = df[["neg_count","dialect","has_positive_emoji","has_negative_emoji"]].values

### **2.3 Combine Features**

In [12]:
X = hstack([X_tfidf, handcrafted])
y = df["label"]

## **3. Dataset Split**
We split the data into:
- 60% training
- 20% validation
- 20% testing

We use stratified splitting to maintain the class distribution across all sets.

### Split Dataset (60-20-20)

In [13]:
# Split into 60/40, 60% training, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

# Split the 40% temp into 20% tesing, 20% validation
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

## **4. Model Training**

#### Naïve Bayes
We use MultinomialNB, which works well for discrete features like word counts or TF-IDF. 
- Alpha = 0.5 is used for Laplace smoothing to avoid zero probabilities.

#### Neural Network (MLP)
We use a feed-forward neural network:
- Hidden layers: (128, 64) → two layers with decreasing size for hierarchical feature abstraction.
- Learning rate: 0.001 → standard starting point.
- Max iterations: 100 → limited to avoid long training times. Early stopping can be added for better efficiency.

### **4.1 Naïve Bayes**

In [14]:
nb = MultinomialNB(alpha=0.5)
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

### **4.2 Neural Network**

In [15]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    learning_rate_init=0.001,
    max_iter=60,
    random_state=42
)

mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

## **5. Model Evaluation**
We evaluate models using three metrics:

- **Accuracy**: overall fraction of correct predictions.
- **Classification Report**: includes **precision**, **recall**, and **F1-score** per class.
- **Confusion Matrix**: raw counts of predicted vs actual labels.

### **5.1 Evaluate Naïve Bayes**

In [16]:
# Accuracy
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f"Naïve Bayes Accuracy: {acc_nb:.2f}")

# Classification report
print("Naïve Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb, target_names=["NEG","OBJ","POS"]))

# Confusion matrix
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=["NEG","OBJ","POS"])
print("Naïve Bayes Confusion Matrix:")
print(cm_nb)

Naïve Bayes Accuracy: 0.75
Naïve Bayes Classification Report:
              precision    recall  f1-score   support

         NEG       0.44      0.07      0.11       337
         OBJ       0.76      0.98      0.86      1505
         POS       0.50      0.02      0.04       160

    accuracy                           0.75      2002
   macro avg       0.57      0.35      0.34      2002
weighted avg       0.68      0.75      0.67      2002

Naïve Bayes Confusion Matrix:
[[  22  314    1]
 [  27 1476    2]
 [   1  156    3]]


### **5.2 Evaluate Neural Network**

In [17]:
# Accuracy
acc_mlp = accuracy_score(y_test, y_pred_mlp)
print(f"MLP Accuracy: {acc_mlp:.2f}")

# Classification report
print("MLP Classification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=["NEG","OBJ","POS"]))

# Confusion matrix
cm_mlp = confusion_matrix(y_test, y_pred_mlp, labels=["NEG","OBJ","POS"])
print("MLP Confusion Matrix:")
print(cm_mlp)

MLP Accuracy: 0.68
MLP Classification Report:
              precision    recall  f1-score   support

         NEG       0.33      0.35      0.34       337
         OBJ       0.80      0.80      0.80      1505
         POS       0.26      0.21      0.24       160

    accuracy                           0.68      2002
   macro avg       0.46      0.46      0.46      2002
weighted avg       0.68      0.68      0.68      2002

MLP Confusion Matrix:
[[ 118  200   19]
 [ 221 1208   76]
 [  20  106   34]]
